In [ ]:
# import libraries
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.data.load_data import load_raw_data, encode_target
from src.data.clean_data import report_unknown_categories

In [ ]:
data = encode_target(load_raw_data('../data/raw/bank-full.csv'))
print(data.head())
print(f'{data.shape[0]} rows and {data.shape[1]} columns')

NameError: name 'encode_target' is not defined

In [ ]:
# quality checks
report_unknown_categories(data)
print('Data quality checks completed successfully.')

In [ ]:
print("Duplicate rows: ", data.duplicated().sum())
print("Missing values: ", data.isnull().sum().sum())
print("Data types: ", data.dtypes)
print("Summary statistics: ", data.describe())
print("Target variable distribution: ", data['y'].value_counts(normalize=True))

In [ ]:
# Univariate: numeric feature distributions
numeric_cols = ['age', 'balance', 'campaign', 'pdays', 'previous']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(data[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Univariate: categorical frequency
categorical_cols = ['job', 'education', 'month', 'contact']
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flat, categorical_cols):
    data[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate: subscription rate by job / education / month
# seasonality
for col in ['job', 'education', 'month']:
    print(f"Subscription rate by {col}:")
    print(data.groupby(col)['y'].mean().sort_values(ascending=False))

In [ ]:
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
rate_by_month = data.groupby('month')['y'].mean().reindex(month_order)
rate_by_month.plot(kind='bar', figsize=(10, 6), title='Subscription Rate by Month')
plt.ylabel('Subscription Rate')
plt.show()

In [ ]:
# Bivariate: balance / age by subscription outcome
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(x='y', y='balance', data=data, ax=axes[0])
sns.boxplot(x='y', y='age', data=data, ax=axes[1])
axes[0].set_title('Balance by Subscription Outcome')
axes[1].set_title('Age by Subscription Outcome')
plt.tight_layout()
plt.show()

In [ ]:
# Multivariate: correlation heatmap
corr = data[['age', 'balance', 'campaign', 'pdays', 'previous', 'day']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Outlier scan (IQR rule)
for col in ['balance', 'campaign']:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((data[col] < lower_bound) | (data[col] > upper_bound)).sum()
    print(f"")